# Benchmarker — Josh's tracker vs. proofread ground truth

Mirrors Eric's `luc3d-bench` evaluation harness (`/Volumes/talmo/eric/luc3d-bench/scripts/evaluate.py`) so the numbers it prints are directly comparable to the rows in `outputs/PAF_3d_kalman/metrics/per_camera_session_metrics.csv`.

**Pipeline**

1. Load the local lucid_lite session.
2. Locate the proofread `*.analysis.h5` GT files alongside each camera's video.
3. Run `josh_source.tracker.SingleFrameTrack` on every frame and collect the per-frame `Group` objects.
4. Stitch identities across frames so each animal has a stable identity id (Hungarian on (cam, track_idx) overlap with the previous frame).
5. Per camera, build bboxes-from-keypoints for both GT and predictions, feed them into a `motmetrics.MOTAccumulator` with the identical IoU≥0.5 rule Eric uses.
6. Report **num_switches**, IDF1, IDP, IDR, MOTA, MOTP per camera + session-mean.
7. (Optional) save assignments as Eric's per-session JSON so they can be dropped into his bench.

In [ ]:
%load_ext autoreload
%autoreload 2
%gui qt

import json
import os
import sys
import time
from collections import defaultdict
from pathlib import Path

import h5py
import motmetrics as mm
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm

# motmetrics 1.4.0 calls np.asfarray, removed in numpy 2.0. Shim it.
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

REPO_ROOT = Path(os.path.expanduser("~/Documents/talmolab/repos/lucid_lite"))
GUI_SOURCE = REPO_ROOT / "gui_source"
if str(GUI_SOURCE) not in sys.path:
    sys.path.insert(0, str(GUI_SOURCE))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from gui_source import main as gui_main
from josh_source import tracker

## 1. Load the session

Same pattern as `tracking_test.ipynb` — launches the GUI so `session.frame_group(idx)` is wired up. The `SESSION_PATH` is the lucid project folder containing `project.slp` + per-camera video subfolders.

In [ ]:
SESSION_PATH = Path("/Users/joshuapark/Documents/talmolab/lucid_folders/10072022145420_small")
SESSION_ID = SESSION_PATH.name.split("_")[0]  # "10072022145420"

app, window = gui_main.main(["main.py", str(SESSION_PATH)])
session = window.session

CAMERAS = [c.name for c in session.cameras]
FRAME_LO, FRAME_HI = session.min_frame, session.max_frame
N_FRAMES = FRAME_HI - FRAME_LO + 1
print(f"session={SESSION_ID}  cameras={CAMERAS}  frames=[{FRAME_LO}..{FRAME_HI}]  ({N_FRAMES} frames)")
print(f"skeleton nodes ({len(session.skeleton.nodes)}): {[n.name for n in session.skeleton.nodes]}")

## 2. Discover proofread GT H5s

Eric's master sheet (`/Volumes/talmo/eric/luc3d-bench/outputs/predictions_master_sheet.tsv`) holds the canonical `<cam>_proofread_h5` paths, but for this session the same `.analysis.h5` files already sit alongside the local videos (one per cam folder, suffix `.predictions.proofread.slp.analysis.h5`). We auto-discover them — falls back to the SMB master sheet if the local copy is missing.

In [ ]:
def find_proofread_h5(session_dir: Path, cam: str) -> Path | None:
    cam_dir = session_dir / cam
    if not cam_dir.is_dir():
        return None
    candidates = sorted(cam_dir.glob("*.predictions.proofread.slp.analysis.h5"))
    return candidates[0] if candidates else None

BENCH_CAMERAS = [c for c in ("back", "backL", "mid", "midL", "top", "topL") if c in CAMERAS]
gt_h5_paths = {cam: find_proofread_h5(SESSION_PATH, cam) for cam in BENCH_CAMERAS}

for cam, p in gt_h5_paths.items():
    status = "OK" if (p and p.exists()) else "MISSING"
    print(f"  {cam:>6}: {status:7}  {p}")

missing = [c for c, p in gt_h5_paths.items() if not (p and p.exists())]
if missing:
    raise FileNotFoundError(f"no proofread H5 for cameras {missing}")

## 3. Load GT tracks + occupancy

Same loader Eric uses: `tracks` is `(n_tracks, 2, n_nodes, n_frames)` → transpose to `(n_frames, n_tracks, n_nodes, 2)`; `track_occupancy` is `(n_frames, n_tracks)` bool. Per-cam dict so the eval loop is trivial.

In [ ]:
def load_gt(proofread_h5: Path):
    with h5py.File(proofread_h5, "r") as f:
        tr = f["tracks"][:]                # (n_tracks, 2, n_nodes, n_frames)
        occ = f["track_occupancy"][:]      # (n_frames, n_tracks)
        track_names = [n.decode() if isinstance(n, bytes) else n for n in f["track_names"][:]]
    tr = np.transpose(tr, (3, 0, 2, 1)).astype(np.float64)  # (n_frames, n_tracks, n_nodes, 2)
    return tr, occ.astype(bool), track_names

gt_by_cam: dict[str, tuple[np.ndarray, np.ndarray, list[str]]] = {}
for cam, p in gt_h5_paths.items():
    tr, occ, names = load_gt(p)
    gt_by_cam[cam] = (tr, occ, names)
    print(f"  {cam:>6}: tracks={tr.shape}  occupancy={occ.shape}  track_names={names}")

N_GT_FRAMES = min(occ.shape[0] for _, occ, _ in gt_by_cam.values())
N_GT_TRACKS = gt_by_cam[BENCH_CAMERAS[0]][0].shape[1]
EVAL_FRAMES = list(range(min(N_FRAMES, N_GT_FRAMES)))
print(f"\nGT tracks per cam = {N_GT_TRACKS},  eval over {len(EVAL_FRAMES)} frames")

## 4. Run `SingleFrameTrack` over every frame

Same construction site as `tracking_test.ipynb`. We keep the full `sft` per frame so we can inspect `sft.groups`, `sft.valid`, and `sft.adjacency_matrix` later if the tracker disagrees with GT somewhere interesting.

In [ ]:
NODE_WEIGHTS = {
    "Nose": 0.7,  "Ear_R": 0.7,  "Ear_L": 0.7,
    "TTI":  1.0,  "TailTip": 0.0,
    "Head": 1.0,  "Trunk": 0.8,
    "Tail_0": 0.0, "Tail_1": 0.0, "Tail_2": 0.0,
    "Shoulder_left":  0.7, "Shoulder_right": 0.7,
    "Haunch_left":    0.7, "Haunch_right":   0.7,
    "Neck": 0.7,
}

proj_cache = tracker.ProjCache()
frames: list = [None] * len(EVAL_FRAMES)
tracker_errors: list[tuple[int, str]] = []

t0 = time.time()
for i, fi in enumerate(tqdm(EVAL_FRAMES, desc="track")):
    fg = session.frame_group(fi)
    if fg is None:
        tracker_errors.append((fi, "no FrameGroup"))
        continue
    try:
        sft = tracker.SingleFrameTrack(fg, session.cameras, node_weights=NODE_WEIGHTS, proj_cache=proj_cache)
    except Exception as e:
        tracker_errors.append((fi, f"{type(e).__name__}: {e}"))
        continue
    frames[i] = sft
elapsed = time.time() - t0

n_ok    = sum(f is not None for f in frames)
n_valid = sum(getattr(f, "valid", False) for f in frames if f is not None)
print(f"\ntracked {n_ok}/{len(EVAL_FRAMES)} frames in {elapsed:.2f}s  ({elapsed/max(n_ok,1)*1000:.1f} ms/frame)")
print(f"  valid (no per-cam dupes in any group): {n_valid}")
print(f"  tracker errors: {len(tracker_errors)}  first 5: {tracker_errors[:5]}")

## 5. Stitch identities across frames

`SingleFrameTrack` reports a list of `Group`s per frame, but the group-index → identity mapping is **per-frame independent** — group 0 in frame K isn't guaranteed to be the same animal as group 0 in frame K+1. For Eric's switches/IDF1 to be meaningful, we need a session-stable identity id per group.

**Approach:** Hungarian assignment on (cam, track_idx) overlap with the previous frame.
- Keep a running map `(cam, track_idx) → identity_id`.
- For each frame K, build a cost matrix between the K groups and the existing identities, where `cost[g, i] = -|members of g that already map to i|`.
- `linear_sum_assignment` picks the best one-to-one assignment.
- Groups with zero overlap to every existing identity are assigned fresh ids (mirrors `next_palette_color` in lucid_lite).
- After assignment, update the map for every (cam, track_idx) in each group.

This handles SLEAP's per-cam track-id stability for free: as long as a per-cam track keeps its index across frames, its group's identity stays put.

In [ ]:
def stitch_identities(frames, eval_frames):
    """Returns per_frame_assignments: list aligned to eval_frames; each entry is
    dict[(cam, track_idx)] -> identity_id. Also returns next_id (== total #identities)."""
    cam_track_to_id: dict[tuple[str, int], int] = {}
    per_frame: list[dict[tuple[str, int], int]] = []
    next_id = 0

    for sft in frames:
        if sft is None or not sft.groups:
            per_frame.append({})
            continue
        groups = sft.groups

        # Existing-identity columns: every id that appears in cam_track_to_id.
        ident_cols = sorted(set(cam_track_to_id.values()))
        col_of_id = {ident: c for c, ident in enumerate(ident_cols)}

        # Overlap matrix: rows=groups, cols=existing identities.
        overlap = np.zeros((len(groups), len(ident_cols)), dtype=np.int64)
        for g_idx, g in enumerate(groups):
            for cam, ti in g.cam_track:
                prev = cam_track_to_id.get((cam, ti))
                if prev is not None and prev in col_of_id:
                    overlap[g_idx, col_of_id[prev]] += 1

        # Hungarian (maximize overlap == minimize negative overlap).
        assigned_ident = [None] * len(groups)
        if overlap.size and overlap.max() > 0:
            r, c = linear_sum_assignment(-overlap)
            for ri, ci in zip(r, c):
                if overlap[ri, ci] > 0:
                    assigned_ident[ri] = ident_cols[ci]

        # Groups that didn't latch onto any prior identity → fresh ids.
        for g_idx in range(len(groups)):
            if assigned_ident[g_idx] is None:
                assigned_ident[g_idx] = next_id
                next_id += 1

        # Write per-frame assignments and refresh the sticky map.
        frame_assignments: dict[tuple[str, int], int] = {}
        for g_idx, g in enumerate(groups):
            ident = assigned_ident[g_idx]
            for cam, ti in g.cam_track:
                frame_assignments[(cam, ti)] = ident
                cam_track_to_id[(cam, ti)] = ident
        per_frame.append(frame_assignments)

    return per_frame, next_id

per_frame_assignments, total_identities = stitch_identities(frames, EVAL_FRAMES)
print(f"total unique identities created across the session: {total_identities}")
ident_counts = defaultdict(int)
for fa in per_frame_assignments:
    for ident in set(fa.values()):
        ident_counts[ident] += 1
top = sorted(ident_counts.items(), key=lambda kv: -kv[1])[:8]
print("top identities by frames-present:", top)

## 6. Per-camera evaluation with `motmetrics`

Identical logic to Eric's `eval_camera` (`/Volumes/talmo/eric/luc3d-bench/scripts/evaluate.py` lines 82–182): for each frame, build `[xmin, ymin, xmax, ymax]` boxes from GT keypoints (5 px pad, ≥3 finite nodes) and from the camera's predicted instances using their lucid_lite track_idx as the per-cam slot. Then `motmetrics.distances.iou_matrix` with `max_iou = 1 - 0.5` and `MOTAccumulator.update(gt_ids, pr_ids, dist)`. Same metric list Eric reports.

In [ ]:
def bbox_from_kpts(kpts: np.ndarray, pad: float = 5.0, min_valid: int = 3):
    valid = np.isfinite(kpts[:, 0]) & np.isfinite(kpts[:, 1])
    if int(valid.sum()) < min_valid:
        return None
    xs, ys = kpts[valid, 0], kpts[valid, 1]
    return np.array([xs.min() - pad, ys.min() - pad, xs.max() + pad, ys.max() + pad], dtype=np.float64)

def kpts_for_predicted_instance(fg, cam: str, track_idx: int):
    for inst in fg.get_instances(cam):
        if inst.track_idx == track_idx:
            return np.array(
                [(p if p is not None else (np.nan, np.nan)) for p in inst.points],
                dtype=np.float64,
            )
    return None

METRIC_COLS = [
    "num_switches", "num_fragmentations",
    "idf1", "idp", "idr", "idtp", "idfp", "idfn",
    "mota", "motp",
    "precision", "recall",
    "num_false_positives", "num_misses",
    "num_detections", "num_objects", "num_predictions",
    "mostly_tracked", "partially_tracked", "mostly_lost",
    "num_unique_objects", "num_frames",
]

def eval_camera(
    cam: str, gt_tracks: np.ndarray, gt_occ: np.ndarray,
    per_frame_assignments, eval_frames, iou_thresh: float = 0.5,
):
    acc = mm.MOTAccumulator(auto_id=True)
    for i, fi in enumerate(eval_frames):
        # GT boxes for this frame.
        gt_boxes, gt_ids = [], []
        for t in range(gt_tracks.shape[1]):
            if not gt_occ[fi, t]:
                continue
            b = bbox_from_kpts(gt_tracks[fi, t])
            if b is None:
                continue
            gt_boxes.append(b)
            gt_ids.append(int(t))

        # Predicted boxes for this cam at this frame, identity from stitched map.
        fg = session.frame_group(fi)
        pr_boxes, pr_ids = [], []
        if fg is not None:
            assigns = per_frame_assignments[i]
            seen_tracks: set[int] = set()
            for inst in fg.get_instances(cam):
                if inst.track_idx is None or inst.track_idx in seen_tracks:
                    continue
                seen_tracks.add(inst.track_idx)
                kpts = kpts_for_predicted_instance(fg, cam, inst.track_idx)
                b = bbox_from_kpts(kpts) if kpts is not None else None
                if b is None:
                    continue
                ident = assigns.get((cam, inst.track_idx), -1)
                if ident < 0:
                    continue
                pr_boxes.append(b)
                pr_ids.append(int(ident))

        gt_np = np.array(gt_boxes) if gt_boxes else np.empty((0, 4))
        pr_np = np.array(pr_boxes) if pr_boxes else np.empty((0, 4))
        dist = mm.distances.iou_matrix(gt_np, pr_np, max_iou=1.0 - iou_thresh)
        acc.update(gt_ids, pr_ids, dist)

    mh = mm.metrics.create()
    summary = mh.compute(acc, metrics=METRIC_COLS, name=cam)
    return summary, acc

per_cam_summaries: dict[str, pd.DataFrame] = {}
for cam in BENCH_CAMERAS:
    tr, occ, _ = gt_by_cam[cam]
    summary, _ = eval_camera(cam, tr, occ, per_frame_assignments, EVAL_FRAMES)
    per_cam_summaries[cam] = summary
    print(f"  {cam:>6}: switches={int(summary.loc[cam, 'num_switches'])}  "
          f"idf1={summary.loc[cam, 'idf1']:.4f}  "
          f"idp={summary.loc[cam, 'idp']:.4f}  "
          f"idr={summary.loc[cam, 'idr']:.4f}  "
          f"mota={summary.loc[cam, 'mota']:.4f}")

## 7. Session-level summary + comparison to Eric's trackers

Same shape as a row group in `outputs/PAF_3d_kalman/metrics/per_camera_session_metrics.csv`. Switches summed across cameras (NOT averaged) so it lines up with Eric's per-session totals; identity-level metrics are camera-mean (his per_camera CSV averages cam-rows when rolled up to per-tracker headlines).

In [ ]:
rows = []
for cam, summary in per_cam_summaries.items():
    r = {"camera": cam}
    r.update(summary.loc[cam].to_dict())
    rows.append(r)
per_cam_df = pd.DataFrame(rows).set_index("camera")

headline_cols = ["num_switches", "num_fragmentations", "idf1", "idp", "idr", "mota", "motp",
                 "precision", "recall", "num_objects", "num_predictions"]
print("per-camera:")
print(per_cam_df[headline_cols].round(4).to_string())

agg = pd.Series({
    "switches_total":   int(per_cam_df["num_switches"].sum()),
    "switches_per_cam": float(per_cam_df["num_switches"].mean()),
    "frags_total":      int(per_cam_df["num_fragmentations"].sum()),
    "idf1_cam_mean":    float(per_cam_df["idf1"].mean()),
    "idp_cam_mean":     float(per_cam_df["idp"].mean()),
    "idr_cam_mean":     float(per_cam_df["idr"].mean()),
    "mota_cam_mean":    float(per_cam_df["mota"].mean()),
    "motp_cam_mean":    float(per_cam_df["motp"].mean()),
    "n_frames":         len(EVAL_FRAMES),
    "n_cameras":        len(BENCH_CAMERAS),
})
print("\nsession headline:")
print(agg.round(4).to_string())

# Eric's numbers for the prior-best on the full 444-cam-session bench (handoff doc table):
ERIC = pd.DataFrame({
    "sleap":                              {"switches/cam": 8.13,  "idf1": 0.6614, "idp": 0.8434},
    "luc3d (baseline)":                   {"switches/cam": 8.36,  "idf1": 0.7360, "idp": 0.9426},
    "luc3d_paf_l1_tracklet (prior best)": {"switches/cam": 4.00,  "idf1": 0.7284, "idp": 0.9575},
    "sleap_nn_filtered (new SOTA)":       {"switches/cam": 2.87,  "idf1": 0.7707, "idp": 0.9347},
    "liezl_cross_view (idf1 leader)":     {"switches/cam": 3.84,  "idf1": 0.8029, "idp": 0.9604},
}).T
print("\nEric's per-session means across his 444-cam bench (handoff doc):")
print(ERIC.to_string())
print("\n^ those are bench-wide averages over 444 cam-sessions; the row above is one session (this one).")

## 8. (Optional) export Eric-format assignments JSON

Writes a single per-session JSON shaped exactly like the entries in `/Volumes/talmo/eric/luc3d-bench/outputs/luc3d_results/<session>.json`. If you drop this into `outputs/luc3d_results/` on Eric's bench machine and rerun `scripts/evaluate.py --filter-session 10072022145420`, the numbers should match what this notebook printed (up to floating-point).

In [ ]:
OUT_DIR = REPO_ROOT / "notebooks" / "benchmark_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

frames_json = []
for i, fi in enumerate(EVAL_FRAMES):
    assigns = per_frame_assignments[i]
    if not assigns:
        continue
    frames_json.append({
        "frame": int(fi),
        "assignments": [[f"{cam}:{ti}", int(ident)] for (cam, ti), ident in assigns.items()],
    })

payload = {
    "sessionIdx": 0,
    "numAnimals": int(N_GT_TRACKS),
    "frames": frames_json,
}
out_path = OUT_DIR / f"{SESSION_ID}.json"
with open(out_path, "w") as f:
    json.dump(payload, f)
print(f"wrote {out_path}  ({len(frames_json)} frames, {sum(len(e['assignments']) for e in frames_json)} assignments)")

out_csv = OUT_DIR / f"{SESSION_ID}_per_camera_metrics.csv"
per_cam_df.reset_index().to_csv(out_csv, index=False)
print(f"wrote {out_csv}")